# Multiclass Simulator Sanity Check

This notebook validates the heterogeneous queue simulation engine before we run more advanced policy experiments.

The goal is to confirm that the multiclass simulation behaves correctly under simple and interpretable policies.

In particular we want to verify that:

• multiple customer classes are simulated correctly  
• class-priority scheduling works as intended  
• abandonment stabilizes overloaded systems  
• flow conservation holds  
• the simulator produces sensible class-level statistics  

## Imports

We load the multiclass simulation engine together with distributions, policies, and helper functions used to summarize simulation results.

The key objects are:

• `CustomerClass` – defines arrival, service, and patience processes for a class  
• `MultiSimConfig` – defines the full simulation configuration  
• `run_sim_multi` – runs the multiclass simulation  
• `PriorityFCFS` – strict class-priority scheduling  
• `MostlyFCFS` – FCFS with a TIQ split inside one class  

We also define a small helper function `class_level_summary` that converts simulation output into a compact summary table.

In [1]:
# Imports

import numpy as np
import pandas as pd

from gi_gi_n_gi_multiclass.core.engine_multi import run_sim_multi
from gi_gi_n_gi_multiclass.core.types_multi import CustomerClass, MultiSimConfig

from gi_gi_n_gi_multiclass.dists.common import Exponential, LogNormal, Weibull
from gi_gi_n_gi_multiclass.outputs.schemas import customers_to_dataframe
from gi_gi_n_gi_multiclass.metrics.steady_state import time_average_L

from gi_gi_n_gi_multiclass.policies.priority_fcfs import PriorityFCFS
from gi_gi_n_gi_multiclass.policies.mostly_fcfs import MostlyFCFS

from gi_gi_n_gi_multiclass.outputs.tables import class_level_summary

## System configuration

We construct a simple two-class overloaded Erlang-A system.

Both classes have exponential arrivals, service, and patience times.

System parameters:

Number of servers  
$N = 10$

Class 0:
- arrival rate $ \lambda_0 = 8 $
- service rate $ \mu_0 = 1 $
- patience rate $ \theta_0 = 0.5 $

Class 1:
- arrival rate $ \lambda_1 = 12 $
- service rate $ \mu_1 = 1 $
- patience rate $ \theta_1 = 1.0 $

Total arrival rate:

$$
\lambda = 8 + 12 = 20
$$

Total service capacity:

$$
N\mu = 10
$$

Since $ \lambda > N\mu $, the system is **overloaded**.

Abandonment therefore plays a crucial role in stabilizing the system.

In [2]:
N_SERVERS = 10
warmup_time = 500.0
run_time = 2000.0
seed = 123

# Class 0: "important" class: moderately patient
cls0 = CustomerClass(
    name="class0",
    interarrival=Exponential(rate=8.0),     # lambda_0
    service=Exponential(rate=1.0),          # mu_0
    patience=Exponential(rate=0.5),         # theta_0 (mean 2)
)

# Class 1: "less important" class: less patient (abandons more quickly)
cls1 = CustomerClass(
    name="class1",
    interarrival=Exponential(rate=12.0),    # lambda_1 (total lambda = 20)
    service=Exponential(rate=1.0),          # mu_1
    patience=Exponential(rate=1.0),         # theta_1 (mean 1)
)

classes = [cls0, cls1]

## Policy 1: Strict class priority

The first policy we test is strict priority across classes.

Policy rule:

• class 0 customers always receive service before class 1  
• within each class service is FCFS  

### Expected behavior

Because class 0 has higher priority:

• class 0 should receive a larger share of service capacity  
• class 0 should have lower waiting times  
• class 1 should experience more abandonment  

The simulation should therefore show **asymmetric performance across classes**, reflecting the priority rule.

In [3]:
policy_priority = PriorityFCFS(priority_order=[0, 1])  # always serve class 0 first if nonempty

cfg_priority = MultiSimConfig(
    n_servers=N_SERVERS,
    classes=classes,
    policy=policy_priority,
    seed=seed,
    warmup_time=warmup_time,
    run_time=run_time,
    collect_event_log=False,
)

summ_priority = class_level_summary(run_sim_multi(cfg_priority))
summ_priority

,scope,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,Wq_mean_served,W_mean_served,L_timeavg,Lq_timeavg
0,OVERALL,39907,19808,20077,19.9535,9.904,10.0385,0.493558,1.501695,20.920751,10.925339
1,class_0,16028,13982,2036,8.0140,6.991,1.0180,0.258269,1.263723,9.069561,2.037347
2,class_1,23879,5826,18041,11.9395,2.913,9.0205,1.058234,2.072812,11.838391,8.882732


### Interpreting the output

The summary table reports statistics for:

• the entire system (`OVERALL`)  
• each customer class (`class_0`, `class_1`)

Key quantities include:

$ \hat{\lambda} $  
Estimated arrival rate

served_rate  
Rate of completed services

abandon_rate  
Rate of abandonment

$W_q$  
Mean waiting time among served customers

$L_q$  
Time-average queue length

Under strict priority we expect class 0 to have **shorter waits and lower abandonment** than class 1.

## Policy 2: MostlyFCFS

The second policy introduces a TIQ-style within-class rule.

The structure of the policy is:

1. choose a class using a class selector
2. choose a customer within that class

Here we keep the same cross-class priority rule as before, but allow class 1 to use a TIQ split internally.

Within class 1:

• customers with waiting time above $w_h$ are served first  
• customers with waiting time below $w_l$ follow  
• remaining customers are served last  


In [4]:
policy_mostly = MostlyFCFS(
    class_selector=PriorityFCFS(priority_order=[0, 1]),
    split_class_id=1,
    w_l=0.5,
    w_h=1.5,
)

cfg_mostly = MultiSimConfig(
    n_servers=N_SERVERS,
    classes=classes,
    policy=policy_mostly,
    seed=seed,
    warmup_time=warmup_time,
    run_time=run_time,
    collect_event_log=False,
)

summ_mostly = class_level_summary(run_sim_multi(cfg_mostly))
summ_mostly

,scope,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,Wq_mean_served,W_mean_served,L_timeavg,Lq_timeavg
0,OVERALL,39907,19831,20055,19.9535,9.9155,10.0275,0.446780,1.453847,20.891820,10.897152
1,class_0,16028,13969,2050,8.0140,6.9845,1.0250,0.256984,1.263077,9.061635,2.031176
2,class_1,23879,5862,18005,11.9395,2.9310,9.0025,0.899058,1.908445,11.821161,8.861948


### What should change?

Because cross-class priority remains the same:

• class 0 behavior should remain similar to the previous policy

However, within class 1 the TIQ rule may alter which customers are served first.

This may change:

• mean waiting times within class 1  
• abandonment patterns within class 1

At this stage we do not expect dramatic performance improvements.  
The purpose is simply to confirm that **TIQ-style within-class logic integrates correctly with multiclass scheduling**.

## Policy comparison

To make the comparison easier to interpret we place the two policy summaries side-by-side.

This highlights how system-level and class-level metrics change when the scheduling rule is modified.

In [5]:
# Compare PriorityFCFS vs MostlyFCFS side-by-side

cols = [
    "scope",
    "lambda_hat",
    "served_rate",
    "abandon_rate",
    "Wq_mean_served",
    "W_mean_served",
    "Lq_timeavg",
    "L_timeavg",
]

summ_priority_view = summ_priority[cols].copy()
summ_priority_view["policy"] = "PriorityFCFS"

summ_mostly_view = summ_mostly[cols].copy()
summ_mostly_view["policy"] = "MostlyFCFS(split=class1)"

pd.concat([summ_priority_view, summ_mostly_view], ignore_index=True).sort_values(["scope", "policy"])

,scope,lambda_hat,served_rate,abandon_rate,Wq_mean_served,W_mean_served,Lq_timeavg,L_timeavg,policy
3,OVERALL,19.9535,9.9155,10.0275,0.446780,1.453847,10.897152,20.891820,MostlyFCFS(split=class1)
0,OVERALL,19.9535,9.9040,10.0385,0.493558,1.501695,10.925339,20.920751,PriorityFCFS
4,class_0,8.0140,6.9845,1.0250,0.256984,1.263077,2.031176,9.061635,MostlyFCFS(split=class1)
1,class_0,8.0140,6.9910,1.0180,0.258269,1.263723,2.037347,9.069561,PriorityFCFS
5,class_1,11.9395,2.9310,9.0025,0.899058,1.908445,8.861948,11.821161,MostlyFCFS(split=class1)
2,class_1,11.9395,2.9130,9.0205,1.058234,2.072812,8.882732,11.838391,PriorityFCFS


## Flow balance check

A basic consistency check for queue simulations is the identity

$$
\lambda \approx \text{service completions} + \text{abandonments}.
$$

If the simulator is functioning correctly, this relationship should hold up to small Monte Carlo error.

The final cell checks this balance for each policy.

In [ ]:
# Two simple sanity assertions (should pass up to small Monte Carlo noise)

def check_flow_balance(summ: pd.DataFrame, tol=0.05):
    overall = summ[summ["scope"] == "OVERALL"].iloc[0]
    lhs = overall["lambda_hat"]
    rhs = overall["served_rate"] + overall["abandon_rate"]
    return float(abs(lhs - rhs)), float(lhs), float(rhs)

for name, summ in [("PriorityFCFS", summ_priority), ("MostlyFCFS", summ_mostly)]:
    diff, lhs, rhs = check_flow_balance(summ)
    print(name, " | |lambda_hat - (served+abandon)| =", diff, " | ", lhs, rhs)

PriorityFCFS  | |lambda_hat - (served+abandon)| = 0.010500000000000398  |  19.9535 19.942999999999998
MostlyFCFS  | |lambda_hat - (served+abandon)| = 0.010500000000000398  |  19.9535 19.942999999999998


## Takeaways

This notebook verifies several key properties of the multiclass simulator:

• multiple customer classes interact correctly in a shared server pool  
• class-priority policies produce the expected asymmetric outcomes  
• TIQ-style within-class rules can be embedded within multiclass policies  
• flow conservation holds for the simulated system  

With these checks in place, we can proceed to more substantive experiments involving:

• hazard-driven policy comparisons  
• TIQ threshold optimization  
• Bassamboo-style mTIQ policies.